In [ ]:
# =========================
# Robust SALSA label/theta/phi converter
# Supports: .xlsx, .xls, .csv, .txt
# Output: grouped Excel by label
# =========================

!pip install -q openpyxl xlrd

import pandas as pd
import numpy as np
import os
from google.colab import files

# -------------------------
# 1) Upload file
# -------------------------
uploaded = files.upload()
INPUT_PATH = list(uploaded.keys())[0]

print("Uploaded file:", INPUT_PATH)

# -------------------------
# 2) Read file robustly
# -------------------------
ext = os.path.splitext(INPUT_PATH)[1].lower()

if ext == ".xlsx":
    df = pd.read_excel(INPUT_PATH, engine="openpyxl")

elif ext == ".xls":
    df = pd.read_excel(INPUT_PATH, engine="xlrd")

elif ext in [".csv", ".txt"]:
    # اول با کاما امتحان می‌کند
    try:
        df = pd.read_csv(INPUT_PATH)
    except:
        # اگر با tab جدا شده باشد
        df = pd.read_csv(INPUT_PATH, sep="\t")

else:
    # اگر پسوند مشخص نبود، چند حالت را امتحان می‌کنیم
    try:
        df = pd.read_excel(INPUT_PATH, engine="openpyxl")
        print("Read as xlsx using openpyxl.")
    except:
        try:
            df = pd.read_excel(INPUT_PATH, engine="xlrd")
            print("Read as xls using xlrd.")
        except:
            try:
                df = pd.read_csv(INPUT_PATH)
                print("Read as csv.")
            except:
                df = pd.read_csv(INPUT_PATH, sep="\t")
                print("Read as tab-separated text.")

# حذف ستون‌های اضافی
df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed")]

print("Columns:")
print(df.columns.tolist())

print("First rows:")
print(df.head(10).to_string())


# -------------------------
# 3) Detect columns
# -------------------------
possible_label_names = [
    "label", "Label", "LABEL",
    "class", "Class",
    "speaker", "Speaker",
    "id", "ID"
]

possible_theta_names = [
    "theta", "teta", "tetha",
    "Theta", "Teta", "Tetha",
    "azimuth", "Azimuth",
    "az", "AZ"
]

possible_phi_names = [
    "phi", "fi",
    "Phi", "Fi",
    "elevation", "Elevation",
    "el", "EL"
]

label_col = None
theta_col = None
phi_col = None

for col in df.columns:
    clean_col = str(col).strip()

    if clean_col in possible_label_names:
        label_col = col

    if clean_col in possible_theta_names:
        theta_col = col

    if clean_col in possible_phi_names:
        phi_col = col

# اگر اسم ستون‌ها پیدا نشد، ترتیب سه ستون اول را می‌گیرد
if label_col is None or theta_col is None or phi_col is None:
    print("Column names not fully recognized. Using first three columns as label, theta, phi.")
    label_col = df.columns[0]
    theta_col = df.columns[1]
    phi_col = df.columns[2]

print("Label column:", label_col)
print("Theta column:", theta_col)
print("Phi column:", phi_col)


# -------------------------
# 4) Clean data
# -------------------------
df_clean = df[[label_col, theta_col, phi_col]].copy()
df_clean.columns = ["label", "theta", "phi"]

df_clean["label"] = pd.to_numeric(df_clean["label"], errors="coerce")
df_clean["theta"] = pd.to_numeric(df_clean["theta"], errors="coerce")
df_clean["phi"] = pd.to_numeric(df_clean["phi"], errors="coerce")

df_clean = df_clean.dropna(subset=["label", "theta", "phi"]).copy()
df_clean["label"] = df_clean["label"].astype(int)

print("Valid rows:", len(df_clean))
print("Labels found:", sorted(df_clean["label"].unique().tolist()))


# -------------------------
# 5) Convert theta/phi to audio coordinates
# -------------------------
theta_rad = np.deg2rad(df_clean["theta"].astype(float))
phi_rad = np.deg2rad(df_clean["phi"].astype(float))

# سه‌بعدی استاندارد
df_clean["x_audio"] = np.cos(phi_rad) * np.cos(theta_rad)
df_clean["y_audio"] = np.cos(phi_rad) * np.sin(theta_rad)
df_clean["z_audio"] = np.sin(phi_rad)

# دوبعدی مناسب برای fusion تصویری
df_clean["x_audio_2d"] = np.cos(phi_rad) * np.sin(theta_rad)
df_clean["y_audio_2d"] = np.sin(phi_rad)

df_clean = df_clean.sort_values(by=["label"]).reset_index(drop=True)

print("Converted preview:")
print(df_clean.head(20).to_string())


# -------------------------
# 6) Save grouped Excel
# -------------------------
base_name = os.path.splitext(os.path.basename(INPUT_PATH))[0]
OUTPUT_PATH = f"/content/{base_name}_grouped_by_label_converted.xlsx"

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_clean.to_excel(writer, sheet_name="all_labels", index=False)

    for label_value, group_df in df_clean.groupby("label"):
        sheet_name = f"label_{label_value}"[:31]
        group_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Saved output:")
print(OUTPUT_PATH)

files.download(OUTPUT_PATH)

Saving 10.csv to 10.csv
Uploaded file: 10.csv
Columns:
['label', 'tetha', 'phi']
First rows:
   label  tetha  phi
0      6   -140   -8
1      6   -140   -8
2      6   -144  -10
3      6   -144  -10
4      6   -144  -11
5      6   -144  -11
6      6   -146   -9
7      6   -146   -9
8      6   -148   -9
9      6   -148   -9
Label column: label
Theta column: tetha
Phi column: phi
Valid rows: 446
Labels found: [0, 3, 5, 6, 9]
Converted preview:
    label  theta  phi   x_audio   y_audio   z_audio  x_audio_2d  y_audio_2d
0       0    120   14 -0.485148  0.840301  0.241922    0.840301    0.241922
1       0    120   14 -0.485148  0.840301  0.241922    0.840301    0.241922
2       0    130   15 -0.620885  0.739942  0.258819    0.739942    0.258819
3       0    130   15 -0.620885  0.739942  0.258819    0.739942    0.258819
4       0    145   22 -0.759505  0.531811  0.374607    0.531811    0.374607
5       0    145   22 -0.759505  0.531811  0.374607    0.531811    0.374607
6       0    133   23 -

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q mediapipe opencv-python pandas openpyxl

import cv2
import os
import numpy as np
import pandas as pd
import mediapipe as mp

from google.colab import files

In [ ]:
uploaded = files.upload()

VIDEO_PATH = list(uploaded.keys())[0]
print("Uploaded video:", VIDEO_PATH)

Saving naardoha.mp4 to naardoha.mp4
Uploaded video: naardoha.mp4


In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("Video could not be opened.")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps else 0

    print("Video path:", VIDEO_PATH)
    print("FPS:", fps)
    print("Total frames:", total_frames)
    print("Duration seconds:", duration)

cap.release()

Video path: naardoha.mp4
FPS: 6.0
Total frames: 368
Duration seconds: 61.333333333333336


In [ ]:
# =========================
# Cell 2: Project settings
# =========================

OUTPUT_DIR = "/content/vision_lip_outputs"
ANNOTATED_DIR = os.path.join(OUTPUT_DIR, "annotated_frames")
EXCEL_PATH = os.path.join(OUTPUT_DIR, "lip_coordinates_3persons_600frames.xlsx")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ANNOTATED_DIR, exist_ok=True)

# تنظیمات هماهنگی با فایل صوتی
TARGET_DURATION_SEC = 60
TARGET_ANALYSIS_FPS = 10
TARGET_TOTAL_FRAMES = TARGET_DURATION_SEC * TARGET_ANALYSIS_FPS  # 600

MAX_PERSONS = 3

# اگر ذخیره فریم‌های علامت‌گذاری‌شده زمان‌بر شد، بعداً False می‌کنیم
SAVE_ANNOTATED_FRAMES = True

print("Video path:", VIDEO_PATH)
print("Output folder:", OUTPUT_DIR)
print("Annotated frames folder:", ANNOTATED_DIR)
print("Excel path:", EXCEL_PATH)

print("Target duration:", TARGET_DURATION_SEC, "sec")
print("Target analysis FPS:", TARGET_ANALYSIS_FPS)
print("Target total frames:", TARGET_TOTAL_FRAMES)
print("First time_sec:", 0)
print("Last time_sec:", (TARGET_TOTAL_FRAMES - 1) / TARGET_ANALYSIS_FPS)

Video path: naardoha.mp4
Output folder: /content/vision_lip_outputs
Annotated frames folder: /content/vision_lip_outputs/annotated_frames
Excel path: /content/vision_lip_outputs/lip_coordinates_3persons_600frames.xlsx
Target duration: 60 sec
Target analysis FPS: 10
Target total frames: 600
First time_sec: 0
Last time_sec: 59.9


In [ ]:
# =========================
# Cell 3A: MediaPipe Tasks setup
# =========================

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from google.colab.patches import cv2_imshow

print("MediaPipe version:", mp.__version__)
print("Has tasks:", hasattr(mp, "tasks"))

# دانلود مدل Face Landmarker
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

MODEL_PATH = "face_landmarker.task"

print("Model downloaded:", MODEL_PATH)

MediaPipe version: 0.10.35
Has tasks: True
Model downloaded: face_landmarker.task


In [ ]:
# =========================
# Cell 3B: Test lip detection on one frame using MediaPipe Tasks
# =========================

import cv2
import numpy as np

# نقاط لب در Face Landmarker / FaceMesh
LIP_LANDMARK_IDS = sorted(set([
    # outer lips
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291,
    185, 40, 39, 37, 0, 267, 269, 270, 409,

    # inner lips
    78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    191, 80, 81, 82, 13, 312, 311, 310, 415
]))

# زمان آزمایشی از ویدیو
TEST_TIME_SEC = 5.0

# ساخت detector
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=3,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

detector = vision.FaceLandmarker.create_from_options(options)

# خواندن یک فریم از ویدیو
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_MSEC, TEST_TIME_SEC * 1000)

success, frame = cap.read()
cap.release()

if not success or frame is None:
    raise ValueError("Frame could not be read. Try another TEST_TIME_SEC.")

height, width, _ = frame.shape

# تبدیل BGR به RGB برای MediaPipe
rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=rgb_frame
)

result = detector.detect(mp_image)

annotated = frame.copy()

if not result.face_landmarks:
    print("No face detected in this frame.")
else:
    print("Detected faces:", len(result.face_landmarks))

    faces_data = []

    for face_index, face_landmarks in enumerate(result.face_landmarks):
        xs = [lm.x * width for lm in face_landmarks]
        ys = [lm.y * height for lm in face_landmarks]

        face_center_x = float(np.mean(xs))
        face_center_y = float(np.mean(ys))

        faces_data.append({
            "face_index": face_index,
            "face_landmarks": face_landmarks,
            "face_center_x": face_center_x,
            "face_center_y": face_center_y
        })

    # مرتب‌سازی سه نفر از چپ به راست
    faces_data = sorted(faces_data, key=lambda x: x["face_center_x"])

    for person_id, face_data in enumerate(faces_data, start=1):
        face_landmarks = face_data["face_landmarks"]

        lip_xs = []
        lip_ys = []

        for lm_id in LIP_LANDMARK_IDS:
            lm = face_landmarks[lm_id]

            x_px = int(lm.x * width)
            y_px = int(lm.y * height)

            lip_xs.append(x_px)
            lip_ys.append(y_px)

            cv2.circle(
                annotated,
                (x_px, y_px),
                2,
                (0, 255, 0),
                -1
            )

        mouth_center_x = int(np.mean(lip_xs))
        mouth_center_y = int(np.mean(lip_ys))

        cv2.circle(
            annotated,
            (mouth_center_x, mouth_center_y),
            5,
            (255, 0, 0),
            -1
        )

        cv2.putText(
            annotated,
            f"P{person_id}",
            (int(face_data["face_center_x"]), int(face_data["face_center_y"])),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

        print(
            f"Person {person_id}: mouth center = "
            f"({mouth_center_x}, {mouth_center_y})"
        )

detector.close()

print("Frame time:", TEST_TIME_SEC, "sec")
cv2_imshow(annotated)

ModuleNotFoundError: No module named 'mediapipe.tasks.c'